In [2]:
import math

import torch

In [2]:
import torch
t = torch.arange(10)
base = 10000
inv_freq = 1.0 / (base ** (torch.arange(0, 768, 2).float() / 768))

In [4]:
inv_freq.shape

torch.Size([384])

In [6]:
inv_freq[None, :, None].shape

torch.Size([1, 384, 1])

In [5]:
import torch
import torch.nn as nn

class RoPEEmbedding(nn.Module):
    '''
    旋转位置编码，此类用来生成作用于K和V的旋转矩阵
    ( cosmth0,     ( sinmth0,
      cosmth1,       sinmth1,
    ...              ...
     cosmth63,     sinmth63,
      cosmth0,     sinmth0,
      cosmth1,      sinmth1,
     ...             ...
      cosmth63,)    sinmth63,)
    m为该token的位置，例如第1个token，第2个token
    thi = 1/ 10000**(2i/d)
    '''
    def __init__(self, base, max_seq_len):
        super().__init__()
        self.base= base
        self.max_seq_len = max_seq_len
    
    def forward(self, x):
        _, seq_len, hidden_size = x.size()
        assert hidden_size%2==0, "hidden_size must be divisiable by 0"
        
        #构造旋转角度向量
        inv_freq = 1.0/self.base**torch.range(0, hidden_size, 2)
        pos = torch.range(0, max(seq_len, self.max_seq_len))
        
        inv_freq = torch.concat([inv_freq, inv_freq], dim=-1)
        emb = torch.einsum("i,j->ij", pos, inv_freq)
        emb_cos = math.cos(emb)
        emb_sin = math.sin(emb)
        
        return emb_cos[:seq_len, ...], emb_sin[:seq_len, ...]
                
        

In [ ]:
import torch
import torch.nn as nn

class RoPEEmbedding_yarn(nn.Module):
    '''
    旋转位置编码，此类用来生成作用于K和V的旋转矩阵
    cosmt, -sinmt
    sinmt, cosmt
    m为该token的位置，例如第1个token，第2个token
    t = 1/ 10000**(2i/d)
    '''
    def __init__(self, base, max_seq_len, base_len):
        super().__init__()
        self.base= base
        self.max_seq_len = max_seq_len
        self.base_len = base_len
    
    def get_ntk_awre(self, true_seq_len, base_seq_len):
        # ntk_alpha = 2**(log2(true_Seq_len/base_len)) -1
        content_value = math.log(true_seq_len/base_seq_len, base=2)
        ntk_alpha = 2**math.ceil(content_value) -1
        return max(ntk_alpha, 1.0)
    def get_mscale(self, scale=1):
        if scale <= 1:
            return 1.0
        return 0.1 * math.log(scale) + 1.0
    def forward(self, x):
        _, seq_len, hidden_size = x.size()
        assert hidden_size%2==0, "hidden_size must be divisiable by 0"
        scaling = self.get_mscale(seq_len/self.base_len)
        #构造旋转角度向量
        inv_freq = 1.0/self.base*self.get_ntk_awre(seq_len, self.base_len)**torch.range(0, hidden_size, 2)
        pos = torch.range(0, max(seq_len, self.max_seq_len))
        
        inv_freq = torch.concat([inv_freq, inv_freq], dim=-1)
        emb = torch.einsum("i,j->ij", pos, inv_freq)
        emb_cos = math.cos(scaling*emb)
        emb_sin = math.sin(scaling*emb)
        
        return emb_cos[:seq_len, ...], emb_sin[:seq_len, ...]
                
        